# SQL to Power BI Handoff

## 1. Purpose
Materialize the approved SQL analyses as eight Power BI-ready CSVs. The three source tables stay separate. CBE rates remain Egypt-only context and are not equivalent to Findex account ownership. Run this notebook from `notebooks/`.

## 2. Connect to SQLite
Use paths relative to the project notebook directory and verify the three source-table counts before changing the database schema.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from pandas.api.types import is_numeric_dtype

processed = Path('../data/processed')
sql_dir = Path('../sql')
output_dir = Path('../powerbi/data')
db_path = processed / 'financial_inclusion.db'
views_sql = sql_dir / '07_powerbi_views.sql'
assert db_path.is_file() and views_sql.is_file()
connection = sqlite3.connect(db_path)
expected_tables = {'findex': 2812, 'imf_fas': 488, 'cbe_context': 34}
source_snapshots = {table: pd.read_sql_query(f'SELECT * FROM {table}', connection)
                    for table in expected_tables}
assert {table: len(df) for table, df in source_snapshots.items()} == expected_tables
print('Source tables:', expected_tables)

Source tables: {'findex': 2812, 'imf_fas': 488, 'cbe_context': 34}


## 3. Create analytical views
`07_powerbi_views.sql` reproduces the approved logic in SQL files 02–06. Rerunning drops and recreates only the eight views.

In [2]:
connection.executescript(views_sql.read_text(encoding='utf-8'))
view_to_file = {
    'vw_findex_market_trends': 'findex_market_trends.csv',
    'vw_findex_egypt_peer_2024': 'findex_egypt_peer_2024.csv',
    'vw_findex_demographic_gaps_2024': 'findex_demographic_gaps_2024.csv',
    'vw_findex_behavior_gaps': 'findex_behavior_gaps.csv',
    'vw_findex_digital_readiness_2024': 'findex_digital_readiness_2024.csv',
    'vw_imf_infrastructure_2024': 'imf_infrastructure_2024.csv',
    'vw_imf_historical_checkpoints': 'imf_historical_checkpoints.csv',
    'vw_cbe_trends': 'cbe_trends.csv',
}
created = {row[0] for row in connection.execute(
    "SELECT name FROM sqlite_master WHERE type = 'view'"
)}
assert created == set(view_to_file), created
print('Views:', sorted(created))

Views: ['vw_cbe_trends', 'vw_findex_behavior_gaps', 'vw_findex_demographic_gaps_2024', 'vw_findex_digital_readiness_2024', 'vw_findex_egypt_peer_2024', 'vw_findex_market_trends', 'vw_imf_historical_checkpoints', 'vw_imf_infrastructure_2024']


## 4. Validate views
Check expected schema, observation keys, critical nulls, numeric values, and nonempty coverage. Historical previous values and first-year changes are intentionally null.

In [3]:
specs = {
    'vw_findex_market_trends': (
        ['country', 'country_role', 'indicator_code', 'indicator_name', 'year', 'value', 'previous_year', 'previous_value', 'change_from_previous_observed_pp'],
        ['country', 'year', 'indicator_code'], ['country', 'indicator_code', 'year', 'value'], ['year', 'value', 'change_from_previous_observed_pp']),
    'vw_findex_egypt_peer_2024': (
        ['indicator_code', 'indicator_name', 'egypt_rate', 'reporting_peers', 'peer_mean', 'egypt_minus_peer_mean_pp'],
        ['indicator_code'], ['indicator_code', 'egypt_rate', 'reporting_peers', 'peer_mean'], ['egypt_rate', 'reporting_peers', 'peer_mean', 'egypt_minus_peer_mean_pp']),
    'vw_findex_demographic_gaps_2024': (
        ['demographic_group', 'comparison', 'indicator_code', 'indicator_name', 'first_value', 'second_value', 'signed_gap_pp', 'gap_magnitude_rank'],
        ['demographic_group', 'indicator_code'], ['demographic_group', 'indicator_code', 'first_value', 'second_value', 'signed_gap_pp'], ['first_value', 'second_value', 'signed_gap_pp', 'gap_magnitude_rank']),
    'vw_findex_behavior_gaps': (
        ['analysis_scope', 'country', 'country_role', 'year', 'demographic_group', 'demographic_subgroup', 'gap_name', 'first_indicator_code', 'second_indicator_code', 'first_value', 'second_value', 'signed_gap_pp'],
        ['analysis_scope', 'country', 'year', 'demographic_group', 'demographic_subgroup', 'gap_name'],
        ['analysis_scope', 'country', 'year', 'demographic_group', 'demographic_subgroup', 'gap_name', 'first_value', 'second_value'], ['year', 'first_value', 'second_value', 'signed_gap_pp']),
    'vw_findex_digital_readiness_2024': (
        ['country', 'country_role', 'year', 'internet_use', 'smartphone_ownership', 'digitally_enabled_account', 'digital_payment_usage'],
        ['country', 'year'], ['country', 'year', 'internet_use', 'smartphone_ownership', 'digitally_enabled_account', 'digital_payment_usage'],
        ['year', 'internet_use', 'smartphone_ownership', 'digitally_enabled_account', 'digital_payment_usage']),
    'vw_imf_infrastructure_2024': (
        ['indicator_name', 'unit', 'country', 'country_role', 'value', 'rank_desc', 'reporting_countries'],
        ['country', 'indicator_name'], ['country', 'country_role', 'indicator_name', 'unit', 'value'], ['value', 'rank_desc', 'reporting_countries']),
    'vw_imf_historical_checkpoints': (
        ['country', 'country_role', 'indicator_name', 'unit', 'year', 'value', 'previous_year', 'previous_value', 'change_from_previous_observed'],
        ['country', 'year', 'indicator_name'], ['country', 'indicator_name', 'unit', 'year', 'value'], ['year', 'value', 'change_from_previous_observed']),
    'vw_cbe_trends': (
        ['metric_code', 'metric_name', 'segment', 'year', 'value', 'unit', 'previous_year', 'year_over_year_change', 'change_unit'],
        ['metric_code', 'year'], ['metric_code', 'year', 'value', 'unit'], ['year', 'value', 'year_over_year_change']),
}
views = {}
for name, (columns, key, critical, numeric) in specs.items():
    frame = pd.read_sql_query(f'SELECT * FROM {name}', connection)
    assert not frame.empty, name
    assert frame.columns.tolist() == columns, name
    assert not frame.duplicated(key).any(), name
    assert frame[critical].notna().all().all(), name
    for column in numeric:
        assert is_numeric_dtype(frame[column]), (name, column)
    views[name] = frame
    print(f'{name}: {len(frame)} rows; columns: {frame.columns.tolist()}')
assert set(views['vw_findex_behavior_gaps'].analysis_scope) == {'national', 'egypt_demographic'}
assert set(views['vw_cbe_trends'].metric_code) == set(source_snapshots['cbe_context'].metric_code)
assert not views['vw_findex_demographic_gaps_2024'].signed_gap_pp.equals(
    views['vw_findex_demographic_gaps_2024'].signed_gap_pp.abs())
print('All view validation checks passed.')

vw_findex_market_trends: 81 rows; columns: ['country', 'country_role', 'indicator_code', 'indicator_name', 'year', 'value', 'previous_year', 'previous_value', 'change_from_previous_observed_pp']
vw_findex_egypt_peer_2024: 4 rows; columns: ['indicator_code', 'indicator_name', 'egypt_rate', 'reporting_peers', 'peer_mean', 'egypt_minus_peer_mean_pp']
vw_findex_demographic_gaps_2024: 28 rows; columns: ['demographic_group', 'comparison', 'indicator_code', 'indicator_name', 'first_value', 'second_value', 'signed_gap_pp', 'gap_magnitude_rank']
vw_findex_behavior_gaps: 84 rows; columns: ['analysis_scope', 'country', 'country_role', 'year', 'demographic_group', 'demographic_subgroup', 'gap_name', 'first_indicator_code', 'second_indicator_code', 'first_value', 'second_value', 'signed_gap_pp']
vw_findex_digital_readiness_2024: 6 rows; columns: ['country', 'country_role', 'year', 'internet_use', 'smartphone_ownership', 'digitally_enabled_account', 'digital_payment_usage']
vw_imf_infrastructure_202

## 5. Equivalence with approved SQL
For seven views, compare every column with the corresponding approved query. The behavior view combines the approved national and Egypt demographic queries into a common schema; compare each scope to its approved source query.

In [4]:
def approved_statements(filename):
    statements, chunk = [], ''
    for line in (sql_dir / filename).read_text(encoding='utf-8').splitlines(True):
        chunk += line
        if sqlite3.complete_statement(chunk):
            statements.append(chunk)
            chunk = ''
    assert not chunk.strip()
    return statements

def compare_rows(actual, expected, key):
    assert actual.columns.tolist() == expected.columns.tolist()
    pd.testing.assert_frame_equal(
        actual.sort_values(key).reset_index(drop=True),
        expected.sort_values(key).reset_index(drop=True),
        check_dtype=False, rtol=1e-12, atol=1e-12,
    )

approved_map = {
    'vw_findex_market_trends': ('02_findex_market_trends.sql', 0),
    'vw_findex_egypt_peer_2024': ('02_findex_market_trends.sql', 1),
    'vw_findex_demographic_gaps_2024': ('03_demographic_gap_analysis.sql', 0),
    'vw_findex_digital_readiness_2024': ('04_financial_behavior_gaps.sql', 2),
    'vw_imf_infrastructure_2024': ('05_imf_infrastructure_analysis.sql', 0),
    'vw_imf_historical_checkpoints': ('05_imf_infrastructure_analysis.sql', 1),
    'vw_cbe_trends': ('06_cbe_egypt_context.sql', 0),
}
for view_name, (filename, position) in approved_map.items():
    approved = pd.read_sql_query(approved_statements(filename)[position], connection)
    if view_name == 'vw_imf_infrastructure_2024':
        roles = source_snapshots['imf_fas'][['country', 'country_role']].drop_duplicates().set_index('country')['country_role']
        approved.insert(3, 'country_role', approved.country.map(roles))
    compare_rows(views[view_name], approved, specs[view_name][1])

national = pd.read_sql_query(approved_statements('04_financial_behavior_gaps.sql')[0], connection)
national = national.rename(columns={'access_code': 'first_indicator_code',
                                    'usage_code': 'second_indicator_code',
                                    'access_value': 'first_value', 'usage_value': 'second_value'})
national.insert(0, 'analysis_scope', 'national')
national.insert(4, 'demographic_group', 'all')
national.insert(5, 'demographic_subgroup', 'all')
national = national[specs['vw_findex_behavior_gaps'][0]]

demographic = pd.read_sql_query(approved_statements('04_financial_behavior_gaps.sql')[1], connection)
first_codes = {'access_to_digital_usage_gap': 'account.t.d',
               'saving_formalization_gap': 'save.any.t.d',
               'borrowing_formalization_gap': 'borrow.any.t.d'}
second_codes = {'access_to_digital_usage_gap': 'g20.any',
                'saving_formalization_gap': 'fin17a.17a1.d',
                'borrowing_formalization_gap': 'fin22a.22a1.22g.d'}
demographic.insert(0, 'analysis_scope', 'egypt_demographic')
demographic.insert(1, 'country', 'Egypt')
demographic.insert(2, 'country_role', 'Main market')
demographic.insert(3, 'year', 2024)
demographic['first_indicator_code'] = demographic.gap_name.map(first_codes)
demographic['second_indicator_code'] = demographic.gap_name.map(second_codes)
demographic = demographic[specs['vw_findex_behavior_gaps'][0]]
expected_behavior = pd.concat([national, demographic], ignore_index=True)
compare_rows(views['vw_findex_behavior_gaps'], expected_behavior,
             specs['vw_findex_behavior_gaps'][1])
print('All eight views match the corresponding approved SQL results.')

All eight views match the corresponding approved SQL results.


## 6. Export Power BI datasets
Each CSV is generated from a SQL view. CSVs are analytical outputs for the later dashboard, not replacements for the processed source tables.

In [5]:
output_dir.mkdir(parents=True, exist_ok=True)
for view_name, filename in view_to_file.items():
    path = output_dir / filename
    views[view_name].to_csv(path, index=False)
    print(f'{filename}: {len(views[view_name])} rows')

findex_market_trends.csv: 81 rows
findex_egypt_peer_2024.csv: 4 rows
findex_demographic_gaps_2024.csv: 28 rows
findex_behavior_gaps.csv: 84 rows
findex_digital_readiness_2024.csv: 6 rows
imf_infrastructure_2024.csv: 25 rows
imf_historical_checkpoints.csv: 71 rows
cbe_trends.csv: 34 rows


## 7. Read-back verification and output summary
Reopen every exported CSV, compare it with the SQL view, and reconfirm the three source tables were not changed.

In [6]:
for view_name, filename in view_to_file.items():
    saved = pd.read_csv(output_dir / filename)
    original = views[view_name]
    assert saved.shape == original.shape
    assert saved.columns.tolist() == original.columns.tolist()
    assert not saved.duplicated(specs[view_name][1]).any()
    assert saved[specs[view_name][2]].notna().all().all()
    pd.testing.assert_frame_equal(saved, original, check_dtype=False,
                                  rtol=1e-12, atol=1e-12)
    assert len(saved) == connection.execute(f'SELECT COUNT(*) FROM {view_name}').fetchone()[0]
for table, original in source_snapshots.items():
    current = pd.read_sql_query(f'SELECT * FROM {table}', connection)
    pd.testing.assert_frame_equal(current, original, check_dtype=False)
    assert len(current) == expected_tables[table]
assert connection.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
connection.close()
print('Eight CSV read-backs match their SQL views. Source table rows: ', expected_tables)

Eight CSV read-backs match their SQL views. Source table rows:  {'findex': 2812, 'imf_fas': 488, 'cbe_context': 34}


### Coverage notes
The views contain only reported observations. UAE is supplementary in Findex comparisons; IMF 2024 deposit and digital indicators have less than six-country coverage. `previous_year` and its change are null for first observed checkpoints. CBE inclusion definitions remain separate from Findex.